<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_FXCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""
FRTB FX Curvature Calculation
"""

# Cell 1: Setup, Initial Portfolio, and Risk Factors (Corresponds to HTML Steps 1 & 3)
# -------------------------------------------------------------------------------------
# This cell loads the initial portfolio data and key regulatory parameters. The output
# displays the starting positions, which corresponds to Steps 1 (identifying risk factors/buckets)
# and 3 (establishing gross positions) in the report.

# Import necessary libraries
import pandas as pd
import numpy as np

# --- Initial Portfolio Data ---
# Gross Curvature Value at Risk (CVR) positions for the FX portfolio.
portfolio_data = [
    {'Bucket': 'KRW^USD', 'Position_ID': 1, 'CVR+': -3378, 'CVR-': 4104},
    {'Bucket': 'KRW^USD', 'Position_ID': 2, 'CVR+': 4330, 'CVR-': -4333},
    {'Bucket': 'HKD^USD', 'Position_ID': 1, 'CVR+': -1575, 'CVR-': -2058},
    {'Bucket': 'HKD^USD', 'Position_ID': 2, 'CVR+': 2574, 'CVR-': 2885}
]

# Create the initial DataFrame
df_gross = pd.DataFrame(portfolio_data)

print("--- Steps 1 & 3: Initial Portfolio and Gross Positions ---")
print("The calculation begins with the initial portfolio positions. Each unique 'Bucket'")
print("represents a distinct risk factor to be aggregated.")
print("\n" + df_gross.to_string(index=False))

--- Steps 1 & 3: Initial Portfolio and Gross Positions ---
The calculation begins with the initial portfolio positions. Each unique 'Bucket'
represents a distinct risk factor to be aggregated.

 Bucket  Position_ID  CVR+  CVR-
KRW^USD            1 -3378  4104
KRW^USD            2  4330 -4333
HKD^USD            1 -1575 -2058
HKD^USD            2  2574  2885


In [2]:
# Cell 2: Calculate Net Positions (Corresponds to HTML Step 4)
# -----------------------------------------------------------------
# As per Article 325g(2), positions exposed to the same risk factor (bucket) must be
# summed to arrive at the net curvature risk positions.

df_net = df_gross.groupby('Bucket')[['CVR+', 'CVR-']].sum().reset_index()

print("\n\n--- Step 4: Net Positions ---")
print("Gross CVR positions for each bucket are summed to get the net positions.")
print("\n" + df_net.to_string(index=False))



--- Step 4: Net Positions ---
Gross CVR positions for each bucket are summed to get the net positions.

 Bucket  CVR+  CVR-
HKD^USD   999   827
KRW^USD   952  -229


In [3]:
# Cell 3: Determine Correlation Parameters (Corresponds to HTML Steps 5 & 6)
# ------------------------------------------------------------------------------
# Per PRA rules, intra-bucket correlation is not applicable as each FX bucket has
# only one risk factor. Cross-bucket correlation for curvature is the square of the
# delta correlation.

# --- Regulatory Parameters ---
GAMMA_DELTA_FX = 0.60
GAMMA_CURVATURE_MEDIUM = GAMMA_DELTA_FX**2

print("\n\n--- Steps 5 & 6: Correlation Parameters ---")
print("Intra-bucket correlation (ρ_kl) is not applicable for the FX risk class.")
print(f"The delta cross-bucket correlation (γ_bc) for FX is specified as {GAMMA_DELTA_FX:.0%}.")
print(f"The curvature cross-bucket correlation is the square of the delta correlation: {GAMMA_DELTA_FX:.2f}^2 = {GAMMA_CURVATURE_MEDIUM:.2f}")



--- Steps 5 & 6: Correlation Parameters ---
Intra-bucket correlation (ρ_kl) is not applicable for the FX risk class.
The delta cross-bucket correlation (γ_bc) for FX is specified as 60%.
The curvature cross-bucket correlation is the square of the delta correlation: 0.60^2 = 0.36


In [4]:
# Cell 4: Calculate Bucket-Level Capital (Corresponds to HTML Step 8)
# --------------------------------------------------------------------
# The capital for each bucket (K_b) is the maximum of the upward and downward
# shock scenarios, floored at zero.

df_bucket_capital = df_net.copy()
df_bucket_capital['K_b+'] = df_bucket_capital['CVR+'].apply(lambda x: max(x, 0))
df_bucket_capital['K_b-'] = df_bucket_capital['CVR-'].apply(lambda x: max(x, 0))
df_bucket_capital['K_b'] = df_bucket_capital[['K_b+', 'K_b-']].max(axis=1)

# Determine the winning scenario for the next step
df_bucket_capital['Selected_Scenario'] = np.where(df_bucket_capital['K_b+'] >= df_bucket_capital['K_b-'], 'Upward', 'Downward')

print("\n\n--- Step 8: Bucket-Level Capital ---")
print("Bucket capital (K_b) is the maximum of the floored upward and downward scenarios.")
print("\n" + df_bucket_capital[['Bucket', 'K_b+', 'K_b-', 'K_b', 'Selected_Scenario']].to_string(index=False))



--- Step 8: Bucket-Level Capital ---
Bucket capital (K_b) is the maximum of the floored upward and downward scenarios.

 Bucket  K_b+  K_b-  K_b Selected_Scenario
HKD^USD   999   827  999            Upward
KRW^USD   952     0  952            Upward


In [5]:
# Cell 5: Determine Bucket Sums (Corresponds to HTML Step 9)
# --------------------------------------------------------------
# The bucket sum (S_b) is the raw CVR value from the winning scenario selected
# in the previous step. This value is not floored at zero and is used for aggregation.

df_bucket_sum = df_bucket_capital.copy()
df_bucket_sum['S_b'] = np.where(df_bucket_sum['Selected_Scenario'] == 'Upward', df_bucket_sum['CVR+'], df_bucket_sum['CVR-'])

print("\n\n--- Step 9: Bucket Sums ---")
print("The bucket sum (S_b) is the net CVR from the most punitive scenario, used for aggregation.")
print("\n" + df_bucket_sum[['Bucket', 'K_b', 'Selected_Scenario', 'S_b']].to_string(index=False))



--- Step 9: Bucket Sums ---
The bucket sum (S_b) is the net CVR from the most punitive scenario, used for aggregation.

 Bucket  K_b Selected_Scenario  S_b
HKD^USD  999            Upward  999
KRW^USD  952            Upward  952


In [6]:
# Cell 6: Calculate Cross-Bucket Capital (Medium Scenario) (Corresponds to HTML Step 10)
# -----------------------------------------------------------------------------------------
# All components are now aggregated to calculate the final risk-class capital charge (RCCR)
# for the medium correlation scenario.

K_b_values = df_bucket_sum['K_b'].values
S_b_values = df_bucket_sum['S_b'].values

# The safeguard function psi(x,y) is 1 because no two S_b values are both negative.
psi = 1 if not (all(s < 0 for s in S_b_values) and len(S_b_values) > 1) else 0

sum_k_sq = np.sum(K_b_values**2)

# Calculate the cross-term
cross_term_sum = 0
for i in range(len(S_b_values)):
    for j in range(i + 1, len(S_b_values)):
        cross_term_sum += 2 * GAMMA_CURVATURE_MEDIUM * S_b_values[i] * S_b_values[j] * psi

rccr_medium = np.sqrt(max(0, sum_k_sq + cross_term_sum))

print("\n\n--- Step 10: Cross-Bucket Capital (Medium Scenario) ---")
print("Aggregating bucket capital using the medium correlation scenario.")
print(f"\n1. Sum of Squares (Σ K_b^2): {sum_k_sq:,.2f}")
print(f"2. Sum of Cross-Products (Σ γ_bc * S_b * S_c): {cross_term_sum:,.2f}")
print("----------------------------------------------------------")
print(f"FX Curvature Capital (Medium): {rccr_medium:,.2f}")



--- Step 10: Cross-Bucket Capital (Medium Scenario) ---
Aggregating bucket capital using the medium correlation scenario.

1. Sum of Squares (Σ K_b^2): 1,904,305.00
2. Sum of Cross-Products (Σ γ_bc * S_b * S_c): 684,754.56
----------------------------------------------------------
FX Curvature Capital (Medium): 1,609.06


In [7]:
# Cell 7: Apply Correlation Scenarios & Determine Final Capital (Corresponds to HTML Step 11)
# ----------------------------------------------------------------------------------------------
# The final capital is the maximum charge resulting from the Medium, High, and Low
# correlation scenarios, as per Article 325h.

# Calculate High and Low Correlations
gamma_high = min(GAMMA_CURVATURE_MEDIUM * 1.25, 1.0)
gamma_low = max(2 * GAMMA_CURVATURE_MEDIUM - 1.0, 0.75 * GAMMA_CURVATURE_MEDIUM)

# Recalculate cross-term for High scenario
cross_term_high = 0
for i in range(len(S_b_values)):
    for j in range(i + 1, len(S_b_values)):
        cross_term_high += 2 * gamma_high * S_b_values[i] * S_b_values[j] * psi
rccr_high = np.sqrt(max(0, sum_k_sq + cross_term_high))

# Recalculate cross-term for Low scenario
cross_term_low = 0
for i in range(len(S_b_values)):
    for j in range(i + 1, len(S_b_values)):
        cross_term_low += 2 * gamma_low * S_b_values[i] * S_b_values[j] * psi
rccr_low = np.sqrt(max(0, sum_k_sq + cross_term_low))

# Final results
scenario_data = {
    'Scenario': ['Medium Correlation', 'High Correlation', 'Low Correlation'],
    'Correlation': [f"{GAMMA_CURVATURE_MEDIUM:.2%}", f"{gamma_high:.2%}", f"{gamma_low:.2%}"],
    'FX Curvature Capital': [rccr_medium, rccr_high, rccr_low]
}
df_scenarios = pd.DataFrame(scenario_data)

final_charge = df_scenarios['FX Curvature Capital'].max()
winning_scenario = df_scenarios.loc[df_scenarios['FX Curvature Capital'].idxmax()]['Scenario']

print("\n\n--- Step 11: Correlation Scenarios & Final Capital ---")
print("The capital is recalculated under stressed correlation assumptions.\n")
print(df_scenarios.to_string(index=False))
print("\n-------------------------------------------------")
print(f" Final FX Curvature Capital Requirement: {final_charge:,.2f}")
print(f" (Driven by the {winning_scenario})")
print("-------------------------------------------------")



--- Step 11: Correlation Scenarios & Final Capital ---
The capital is recalculated under stressed correlation assumptions.

          Scenario Correlation  FX Curvature Capital
Medium Correlation      36.00%           1609.055487
  High Correlation      45.00%           1661.399470
   Low Correlation      27.00%           1554.950456

-------------------------------------------------
 Final FX Curvature Capital Requirement: 1,661.40
 (Driven by the High Correlation)
-------------------------------------------------
